# Extract fields and identify handwriting



This standalone notebook creates a GPT-5.2 Content Understanding analyzer for property-claim fields and a schema-level handwriting judgment for each field. It also runs Document Intelligence Layout on the same local files to report the native OCR `isHandwritten` style signal.



These are different signals: the Content Understanding values are model inferences requested by the custom schema; the Document Intelligence value is handwriting-style metadata. Extraction confidence does not measure whether text is handwritten. No Studio project or storage account is used by this notebook.

In [ ]:
# Install once per environment; rerunning is safe.

%pip install -q "azure-ai-contentunderstanding>=1.1.0" "azure-identity>=1.17.0" "pandas>=2.2.0" "requests>=2.32.0" "ipywidgets>=8.1.0" "python-dotenv>=1.0.0"



from datetime import datetime, timezone

import json

import mimetypes

import os

from pathlib import Path

import re

import time



import ipywidgets as widgets

import pandas as pd

import requests

from IPython.display import display

from azure.ai.contentunderstanding import ContentUnderstandingClient

from azure.core.exceptions import ResourceNotFoundError

from azure.identity import DefaultAzureCredential

from dotenv import load_dotenv



search_roots = [Path.cwd(), *Path.cwd().parents]

REPO_ROOT = next(

    (root for root in search_roots if (root / "data" / "propertyclaims").is_dir()),

    Path.cwd(),

)

for env_path in (

    REPO_ROOT / ".env",

    REPO_ROOT / "apps" / "workshop" / "api" / ".env",

    REPO_ROOT / "demo" / "sov" / ".env",

    REPO_ROOT / "demo" / "handwritten" / ".env",

):

    if env_path.is_file():

        load_dotenv(env_path, override=False)



API_VERSION = "2026-06-01-preview"

DI_API_VERSION = "2024-11-30"

ANALYZER_ID = "propertyClaimHandwritingGpt52V1"

ENDPOINT = (

    os.getenv("APP_CONTENT_UNDERSTANDING_ENDPOINT")

    or os.getenv("CONTENTUNDERSTANDING_ENDPOINT")

    or os.getenv("FOUNDRY_ENDPOINT")

)

GPT52_DEPLOYMENT = os.getenv("GPT52_MODEL_DEPLOYMENT", "gpt-5.2")

EMBEDDING_DEPLOYMENT = os.getenv("EMBEDDING_MODEL_DEPLOYMENT", "text-embedding-3-large")

INPUT_FILES = [REPO_ROOT / "data" / "propertyclaims" / "claimform_handwritten_1.pdf"]

OUTPUT_DIR = REPO_ROOT / "demo" / "handwritten" / "output"

CONFIGURE_DEFAULTS = True

SAVE_RAW_RESPONSES = False

RUN_AZURE = False



print(f"Analyzer: {ANALYZER_ID}")

print(f"Inputs: {[str(path) for path in INPUT_FILES]}")

print(f"Model deployments: gpt-5.2 -> {GPT52_DEPLOYMENT}; text-embedding-3-large -> {EMBEDDING_DEPLOYMENT}")

Note: you may need to restart the kernel to use updated packages.
Analyzer: propertyClaimHandwritingGpt52V1
Inputs: ['c:\\Users\\jomedin\\Documents\\azure-ai-foundry-insurance\\data\\propertyclaims\\claimform_handwritten_1.pdf']
Model deployments: gpt-5.2 -> gpt-5.2; text-embedding-3-large -> text-embedding-3-large


In [2]:
INFORMATION_FIELDS = {

    "PolicyholderFirstName": "Policyholder given name.",

    "PolicyholderLastName": "Policyholder family name.",

    "TelephoneNumber": "Policyholder telephone number.",

    "PolicyNumber": "Unique policy number.",

    "CoverageType": "Coverage type.",

    "ClaimNumber": "Unique claim number.",

    "PolicyEffectiveDate": "Policy effective date.",

    "PolicyExpirationDate": "Policy expiration date.",

    "DamageDeductibleAmount": "Damage deductible amount.",

    "DateOfDamageOrLoss": "Date of damage or loss.",

    "TimeOfLoss": "Time of loss.",

    "DatePrepared": "Date the form was prepared.",

    "PropertyAddress": "Full damaged-property address.",

    "MailingAddress": "Full policyholder mailing address.",

}



def field_with_handwriting(description):

    return {

        "type": "object",

        "method": "generate",

        "description": (

            f"{description} Return the visible value and independently judge whether "

            "that value is handwritten rather than printed or typed."

        ),

        "properties": {

            "Value": {

                "type": "string",

                "method": "generate",

                "description": description,

            },

            "IsHandwritten": {

                "type": "boolean",

                "method": "generate",

                "description": "True only when the field value itself is visibly handwritten.",

            },

            "HandwritingConfidence": {

                "type": "number",

                "method": "generate",

                "description": "A 0 to 1 model judgment for the handwriting classification, separate from extraction confidence.",

            },

            "Evidence": {

                "type": "string",

                "method": "generate",

                "description": "Short visible text evidence supporting the handwriting judgment.",

            },

        },

    }



ANALYZER_OWNER = "handwritten-analyzer-notebook-v1"

ANALYZER_RESOURCE = {

    "baseAnalyzerId": "prebuilt-document",

    "description": "Extract property-claim information and judge handwriting for each field.",

    "config": {

        "returnDetails": True,

        "enableOcr": True,

        "enableLayout": True,

        "estimateFieldSourceAndConfidence": True,

    },

    "fieldSchema": {

        "name": "PropertyClaimHandwritingFields",

        "description": "Property claim values with field-level handwriting judgments.",

        "fields": {

            **{name: field_with_handwriting(description) for name, description in INFORMATION_FIELDS.items()},

            "DocumentContainsHandwriting": {

                "type": "boolean",

                "method": "generate",

                "description": "True when any meaningful filled value in the document is handwritten.",

            },

        },

    },

    "models": {"completion": "gpt-5.2", "embedding": "text-embedding-3-large"},

    "tags": {

        "demo": "handwriting",

        "signal": "schema-inference",

        "owner": ANALYZER_OWNER,

    },

}



assert re.fullmatch(r"[A-Za-z][A-Za-z0-9_-]{2,63}", ANALYZER_ID)

assert all(re.fullmatch(r"[A-Za-z][A-Za-z0-9_]*", name) for name in ANALYZER_RESOURCE["fieldSchema"]["fields"])

print(json.dumps(ANALYZER_RESOURCE, indent=2))

{
  "baseAnalyzerId": "prebuilt-document",
  "description": "Extract property-claim information and judge handwriting for each field.",
  "config": {
    "returnDetails": true,
    "enableOcr": true,
    "enableLayout": true,
    "estimateFieldSourceAndConfidence": true
  },
  "fieldSchema": {
    "name": "PropertyClaimHandwritingFields",
    "description": "Property claim values with field-level handwriting judgments.",
    "fields": {
      "PolicyholderFirstName": {
        "type": "object",
        "method": "generate",
        "description": "Policyholder given name. Return the visible value and independently judge whether that value is handwritten rather than printed or typed.",
        "properties": {
          "Value": {
            "type": "string",
            "method": "generate",
            "description": "Policyholder given name."
          },
          "IsHandwritten": {
            "type": "boolean",
            "method": "generate",
            "description": "True onl

In [3]:
VALUE_KEYS = {

    "string": ("valueString", "value_string"),

    "boolean": ("valueBoolean", "value_boolean"),

    "number": ("valueNumber", "value_number"),

}



def as_dict(value):

    if hasattr(value, "as_dict"):

        return value.as_dict()

    if isinstance(value, dict):

        return value

    raise TypeError(f"Expected an SDK model or dictionary, got {type(value).__name__}")



def analysis_body(payload):

    body = as_dict(payload)

    for key in ("result", "analyzeResult", "analyze_result"):

        nested = body.get(key)

        if isinstance(nested, dict):

            body = nested

    return body



def content_fields(payload, label="Content Understanding"):

    body = analysis_body(payload)

    contents = body.get("contents") or []

    fields = {}

    for content in contents:

        fields.update((as_dict(content).get("fields") or {}))

    if not fields:

        summary = {

            "top_level_keys": sorted(as_dict(payload)),

            "result_keys": sorted(body),

            "content_count": len(contents),

            "warnings": body.get("warnings"),

        }

        raise RuntimeError(f"{label} returned no fields. Response summary: {summary}")

    return fields



def scalar_value(field):

    field = as_dict(field) if field else {}

    for key in VALUE_KEYS.get(str(field.get("type", "")).casefold(), ()):

        if key in field:

            return field[key]

    return field.get("value")



def configure_model_defaults(credential):

    token = credential.get_token("https://cognitiveservices.azure.com/.default").token

    url = f"{ENDPOINT.rstrip('/')}/contentunderstanding/defaults?api-version={API_VERSION}"

    headers = {"Authorization": f"Bearer {token}"}

    current_response = requests.get(url, headers=headers, timeout=30)

    current = current_response.json() if current_response.ok and current_response.text else {}

    deployments = dict(current.get("modelDeployments") or {})

    deployments.update({

        "gpt-5.2": GPT52_DEPLOYMENT,

        "text-embedding-3-large": EMBEDDING_DEPLOYMENT,

    })

    response = requests.patch(

        url,

        headers={**headers, "Content-Type": "application/merge-patch+json"},

        json={"modelDeployments": deployments},

        timeout=30,

    )

    response.raise_for_status()

    return deployments



def poll_json(url, headers, timeout_seconds=300):

    deadline = time.time() + timeout_seconds

    while time.time() < deadline:

        response = requests.get(url, headers=headers, timeout=30)

        response.raise_for_status()

        payload = response.json()

        status = str(payload.get("status", "succeeded")).casefold()

        if status in {"succeeded", "partiallysucceeded"}:

            return payload

        if status in {"failed", "canceled", "cancelled"}:

            raise RuntimeError(json.dumps(payload.get("error", payload), indent=2))

        time.sleep(2)

    raise TimeoutError("Azure analysis did not finish within five minutes.")



def analyze_document_intelligence(path, credential):

    token = credential.get_token("https://cognitiveservices.azure.com/.default").token

    url = (

        f"{ENDPOINT.rstrip('/')}/documentintelligence/documentModels/"

        f"prebuilt-layout:analyze?api-version={DI_API_VERSION}&outputContentFormat=markdown"

    )

    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/octet-stream"}

    response = requests.post(url, headers=headers, data=path.read_bytes(), timeout=60)

    response.raise_for_status()

    if response.status_code == 200:

        payload = response.json()

    else:

        operation_url = response.headers.get("operation-location")

        if not operation_url:

            raise RuntimeError("Document Intelligence did not return an operation URL.")

        payload = poll_json(operation_url, headers)

    return payload.get("analyzeResult", payload.get("result", payload))



def nested_value(field, name):

    field = as_dict(field) if field else {}

    value_object = field.get("valueObject") or field.get("value_object") or {}

    return scalar_value(value_object.get(name))



def native_handwriting(result):

    styles = [

        style for style in result.get("styles", [])

        if style.get("isHandwritten", style.get("is_handwritten"))

    ]

    return {

        "detected": bool(styles),

        "confidence": max((style.get("confidence", 0) for style in styles), default=None),

        "style_count": len(styles),

    }


In [4]:
assert RUN_AZURE, "Review the settings above, then set RUN_AZURE = True."

assert ENDPOINT, "Set APP_CONTENT_UNDERSTANDING_ENDPOINT."

assert INPUT_FILES and all(path.is_file() for path in INPUT_FILES), "Set INPUT_FILES to local PDFs or images."



credential = DefaultAzureCredential()

if CONFIGURE_DEFAULTS:

    configured = configure_model_defaults(credential)

    print("Content Understanding model defaults are ready:", {

        "gpt-5.2": configured["gpt-5.2"],

        "text-embedding-3-large": configured["text-embedding-3-large"],

    })



client = ContentUnderstandingClient(

    endpoint=ENDPOINT,

    credential=credential,

    api_version=API_VERSION,

)

allow_replace = False

try:

    existing = as_dict(client.get_analyzer(analyzer_id=ANALYZER_ID))

except ResourceNotFoundError:

    existing = None

if existing:

    if (existing.get("tags") or {}).get("owner") != ANALYZER_OWNER:

        raise RuntimeError(f"Analyzer {ANALYZER_ID!r} exists but is not owned by this notebook.")

    allow_replace = True



client.begin_create_analyzer(

    analyzer_id=ANALYZER_ID,

    resource=ANALYZER_RESOURCE,

    allow_replace=allow_replace,

).result()

metadata = as_dict(client.get_analyzer(analyzer_id=ANALYZER_ID))

print({

    "analyzer_id": metadata.get("analyzerId", ANALYZER_ID),

    "status": metadata.get("status", "ready"),

    "api_version": API_VERSION,

})

Content Understanding model defaults are ready: {'gpt-5.2': 'gpt-5.2', 'text-embedding-3-large': 'text-embedding-3-large'}
{'analyzer_id': 'propertyClaimHandwritingGpt52V1', 'status': 'ready', 'api_version': '2026-06-01-preview'}


In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_dir = OUTPUT_DIR / "raw"

if SAVE_RAW_RESPONSES:

    raw_dir.mkdir(exist_ok=True)

rows = []



for path in INPUT_FILES:

    content_type = mimetypes.guess_type(path.name)[0] or "application/octet-stream"

    cu_result = as_dict(client.begin_analyze_binary(

        analyzer_id=ANALYZER_ID,

        binary_input=path.read_bytes(),

        content_type=content_type,

    ).result())

    di_result = analyze_document_intelligence(path, credential)

    fields = content_fields(cu_result, f"{ANALYZER_ID} on {path.name}")

    native = native_handwriting(di_result)

    document_judgment = fields.get("DocumentContainsHandwriting") or {}



    for field_name in INFORMATION_FIELDS:

        field = fields.get(field_name) or {}

        rows.append({

            "Document": path.name,

            "Field": field_name,

            "Value": nested_value(field, "Value"),

            "GPT is handwritten": nested_value(field, "IsHandwritten"),

            "GPT handwriting score": nested_value(field, "HandwritingConfidence"),

            "Extraction confidence": as_dict(field).get("confidence") if field else None,

            "Evidence": nested_value(field, "Evidence"),

            "Document handwriting (GPT)": scalar_value(document_judgment),

            "Document handwriting (DI native)": native["detected"],

            "DI native confidence": native["confidence"],

            "Page source": as_dict(field).get("source") if field else None,

        })



    if SAVE_RAW_RESPONSES:

        (raw_dir / f"{path.stem}.json").write_text(

            json.dumps({

                "content_understanding": cu_result,

                "document_intelligence": di_result,

            }, indent=2),

            encoding="utf-8",

        )



results = pd.DataFrame(rows)

if results.empty or results["Value"].notna().sum() == 0:

    first_fields = content_fields(cu_result, ANALYZER_ID)

    sample = {name: as_dict(field) for name, field in list(first_fields.items())[:2]}

    raise RuntimeError(

        "The analyzer returned fields but no nested Value outputs. "

        f"Inspect this field sample: {json.dumps(sample, default=str)[:2000]}"

    )



print(

    f"Extracted {int(results['Value'].notna().sum())} values and "

    f"{int(results['GPT is handwritten'].notna().sum())} handwriting judgments "

    f"from {results['Document'].nunique()} document(s)."

)

display(results)


Extracted 14 values and 14 handwriting judgments from 1 document(s).


,Document,Field,Value,GPT is handwritten,GPT handwriting score,Extraction confidence,Evidence,Document handwriting (GPT),Document handwriting (DI native),DI native confidence,Page source
0,claimform_handwritten_1.pdf,PolicyholderFirstName,Yasmin,False,0.2,None,Yasmin,False,True,1,None
1,claimform_handwritten_1.pdf,PolicyholderLastName,Faisal,False,0.2,None,Faisal,False,True,1,None
2,claimform_handwritten_1.pdf,TelephoneNumber,410-555-0190,False,0.2,None,410-555-0190,False,True,1,None
3,claimform_handwritten_1.pdf,PolicyNumber,PH954738,False,0.2,None,PH954738,False,True,1,None
4,claimform_handwritten_1.pdf,CoverageType,HomeowneR,False,0.2,None,HomeowneR,False,True,1,None
5,claimform_handwritten_1.pdf,ClaimNumber,CL264831,False,0.2,None,CL264831,False,True,1,None
6,claimform_handwritten_1.pdf,PolicyEffectiveDate,2020-02-10,False,0.2,None,2020-02-10,False,True,1,None
7,claimform_handwritten_1.pdf,PolicyExpirationDate,2025-03-31,False,0.2,None,2025-03-31,False,True,1,None
8,claimform_handwritten_1.pdf,DamageDeductibleAmount,$1000,False,0.2,None,$1000,False,True,1,None
9,claimform_handwritten_1.pdf,DateOfDamageOrLoss,2025-01-23,False,0.2,None,2025-01-23,False,True,1,None


In [7]:
review_controls = {}

items = []

for index, row in results.iterrows():

    value = widgets.Text(

        value="" if pd.isna(row["Value"]) else str(row["Value"]),

        description=row["Field"],

        layout=widgets.Layout(width="620px"),

    )

    handwritten = widgets.Checkbox(

        value=bool(row["GPT is handwritten"]) if pd.notna(row["GPT is handwritten"]) else False,

        description="handwritten",

        indent=False,

    )

    review_controls[index] = (value, handwritten)

    items.append(widgets.HBox([value, handwritten]))



save_button = widgets.Button(description="Save reviewed results", button_style="primary")

save_output = widgets.Output()



def save_reviewed(_button):

    reviewed = results.copy()

    for index, (value, handwritten) in review_controls.items():

        reviewed.loc[index, "Reviewed value"] = value.value

        reviewed.loc[index, "Reviewed is handwritten"] = handwritten.value

    reviewed.to_csv(OUTPUT_DIR / "reviewed-handwriting.csv", index=False)

    manifest = {

        "analyzer_id": ANALYZER_ID,

        "api_version": API_VERSION,

        "processed_at": datetime.now(timezone.utc).isoformat(),

        "source_document_ids": [path.name for path in INPUT_FILES],

        "results": reviewed.where(pd.notna(reviewed), None).to_dict(orient="records"),

    }

    (OUTPUT_DIR / "reviewed-handwriting.json").write_text(

        json.dumps(manifest, indent=2), encoding="utf-8"

    )

    with save_output:

        save_output.clear_output()

        print(f"Saved reviewed results to {OUTPUT_DIR.resolve()}")



save_button.on_click(save_reviewed)

display(widgets.VBox(items + [save_button, save_output]))


## Reading the output



Use `Document handwriting (DI native)` as the service-native handwriting indicator. `GPT is handwritten` and `GPT handwriting score` are schema-driven judgments intended for field-level review; they are not the same as extraction confidence and should be evaluated against reviewed handwriting labels before automation.



In the verified sample run, all 14 values were returned. GPT-5.2 classified the individual values as not handwritten with a low `0.2` handwriting score, while Document Intelligence's native document signal returned `True` with confidence `1.0`. That disagreement is visible evidence that a generated handwriting flag should not replace the native OCR style signal.



The reviewed JSON and CSV contain normalized predictions and reviewer changes, but no API keys, access tokens, document bytes, or full OCR content. Full CU and DI responses can contain document text and are written only when `SAVE_RAW_RESPONSES = True`; keep that output outside source control.